# Partial Derivatives

| Difficulty | ███░░░░░░░ 3/10 |
| :--- | :--- |
| Prerequisites | 02.02 |
| Time | ~55 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/02_Calculus/03_partial_derivatives.ipynb)

---

## 🎯 Learning Objective

Extend differentiation to multivariate functions and assemble partial derivatives into the gradient vector.

---

## 📐 Theory

A function of several variables, $f(x_1, \dots, x_n)$, doesn't have a single "slope" — the
rate of change depends on *which direction* you move. The **partial derivative** answers a
restricted question: how does $f$ change if you nudge just one input, holding the rest fixed?

$$\frac{\partial f}{\partial x_i} = \lim_{h \to 0} \frac{f(x_1, \dots, x_i + h, \dots, x_n) - f(x_1, \dots, x_i, \dots, x_n)}{h}$$

This is the single-variable derivative from `02.02`, applied to the function you get by
freezing every variable except $x_i$. To compute $\partial f/\partial x_i$, treat every other
variable as a constant and differentiate w.r.t. $x_i$ with the ordinary rules.

### Example

For $f(x, y) = x^2 y + \sin(y)$: $\frac{\partial f}{\partial x} = 2xy$ (treat $y$ as a
constant, so $x^2 y$ differentiates like $c\cdot x^2$) and
$\frac{\partial f}{\partial y} = x^2 + \cos(y)$ (treat $x$ as a constant, so $x^2 y$
differentiates like a constant times $y$).

### The gradient vector

Stacking all partial derivatives into a single vector gives the **gradient**:

$$\nabla f(x_1, \dots, x_n) = \left(\frac{\partial f}{\partial x_1}, \dots, \frac{\partial f}{\partial x_n}\right)$$

The gradient packages *every* direction's rate-of-change information at once. We'll prove in
`02.05` that $\nabla f$ points toward steepest ascent — but even without that, it's already
doing something crucial: for a function of millions of parameters (a loss function), it tells
you exactly how each individual parameter should move to change the output.

### Geometric picture: level curves and slices

Picture the surface $z=f(x,y)$ above the $xy$-plane. Slicing with $y=y_0$ gives a curve in
$x$; its ordinary slope at $x_0$ is $\partial f/\partial x$ there. Slicing with $x=x_0$ gives
$\partial f/\partial y$. A **level curve** $f(x,y)=c$ is where the surface has constant height
$c$ — walking *along* it gives zero change in $f$, walking *across* it gives the steepest
change. This contour picture is the one you'll use constantly for loss landscapes.

### Mixed partials and Clairaut's theorem

$\frac{\partial^2 f}{\partial x \partial y} = \frac{\partial}{\partial x}\left(\frac{\partial f}{\partial y}\right)$
is a partial derivative of a partial derivative. Clairaut's theorem says that for smooth
functions, order doesn't matter:
$\frac{\partial^2 f}{\partial x \partial y} = \frac{\partial^2 f}{\partial y \partial x}$. This
symmetry reappears in `02.06` as the symmetry of the Hessian matrix.

### Why this matters for training

A network's loss $L(w_1, \dots, w_N)$ depends on every weight simultaneously — often billions
of them. Training needs, for each weight $w_i$, whether nudging it up or down improves the
loss with everything else fixed: exactly $\partial L/\partial w_i$, bundled across all $i$ into
$\nabla L$. Backpropagation (`02.04`, `02.10`) computes all of these at once, without ever
forming each one by brute-force perturbation.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

A contour plot of $f(x,y)$ with the two axis-aligned "slices" drawn on top makes partial
derivatives concrete: each slice is a single-variable function, and its ordinary slope at a
point is the partial derivative in that direction.

In [ ]:
# f(x, y) = x^2 + y^2 -- a bowl. Slice along x=x0 and y=y0 through a point to see each
# partial derivative as an ordinary 1D slope on its respective slice.
f = lambda x, y: x ** 2 + y ** 2
x0, y0 = 1.0, 1.5

x_grid = np.linspace(-2.5, 2.5, 200)
y_grid = np.linspace(-2.5, 2.5, 200)
X, Y = np.meshgrid(x_grid, y_grid)
Z = f(X, Y)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Contour plot with the two slice lines marked
cs = axes[0].contour(X, Y, Z, levels=15, cmap="viridis")
axes[0].axhline(y0, color="#DD8452", lw=2, label=f"slice y={y0} (varies x)")
axes[0].axvline(x0, color="#C44E52", lw=2, label=f"slice x={x0} (varies y)")
axes[0].plot(x0, y0, 'ko', zorder=5)
axes[0].set_xlabel("x"); axes[0].set_ylabel("y"); axes[0].set_title("Contours of f(x,y)=x^2+y^2")
axes[0].legend(fontsize=8, loc="upper left")

# Slice at y=y0: f becomes a function of x alone; its slope at x0 is df/dx
x_line = np.linspace(-2.5, 2.5, 200)
axes[1].plot(x_line, f(x_line, y0), color="#DD8452", lw=2)
df_dx = 2 * x0  # partial derivative w.r.t. x, y held fixed
tangent = f(x0, y0) + df_dx * (x_line - x0)
axes[1].plot(x_line, tangent, 'k--', lw=1, label=f"slope = df/dx = {df_dx:.1f}")
axes[1].plot(x0, f(x0, y0), 'ko')
axes[1].set_xlabel("x"); axes[1].set_ylabel(f"f(x, y={y0})"); axes[1].set_title("Slice at fixed y")
axes[1].legend(fontsize=9)

# Slice at x=x0: f becomes a function of y alone; its slope at y0 is df/dy
y_line = np.linspace(-2.5, 2.5, 200)
axes[2].plot(y_line, f(x0, y_line), color="#C44E52", lw=2)
df_dy = 2 * y0
tangent_y = f(x0, y0) + df_dy * (y_line - y0)
axes[2].plot(y_line, tangent_y, 'k--', lw=1, label=f"slope = df/dy = {df_dy:.1f}")
axes[2].plot(y0, f(x0, y0), 'ko')
axes[2].set_xlabel("y"); axes[2].set_ylabel(f"f(x={x0}, y)"); axes[2].set_title("Slice at fixed x")
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()
print(f"Gradient at (x,y)=({x0},{y0}):  grad f = (df/dx, df/dy) = ({df_dx}, {df_dy})")

## 🐍 Implementation from Scratch

We compute partial derivatives numerically (nudge one coordinate, hold the rest fixed — the
definition, directly), assemble them into a gradient vector, and confirm Clairaut's theorem
(mixed partials commute) on a non-trivial function.

In [ ]:
def partial_derivative(f, point, i, h=1e-6):
    # Nudge only coordinate i, holding the rest fixed -- this IS the definition, not an
    # approximation trick layered on top of it
    point = np.array(point, dtype=float)
    point_plus, point_minus = point.copy(), point.copy()
    point_plus[i] += h
    point_minus[i] -= h
    return (f(*point_plus) - f(*point_minus)) / (2 * h)

def gradient(f, point, h=1e-6):
    # The gradient is nothing more than "apply partial_derivative once per coordinate"
    return np.array([partial_derivative(f, point, i, h) for i in range(len(point))])

# f(x, y) = x^2 y + sin(y)  ->  df/dx = 2xy,  df/dy = x^2 + cos(y)  (derived in Theory)
f = lambda x, y: x**2 * y + np.sin(y)
point = (1.5, 0.5)
numeric_grad = gradient(f, point)
analytic_grad = np.array([2 * point[0] * point[1], point[0]**2 + np.cos(point[1])])

print(f"Numeric gradient  at {point}: {np.round(numeric_grad, 5)}")
print(f"Analytic gradient at {point}: {np.round(analytic_grad, 5)}")
print(f"Match: {np.allclose(numeric_grad, analytic_grad, atol=1e-4)}")

# Clairaut's theorem: mixed partials should commute. Check on g(x,y) = x^3 y^2 + e^(xy)
def mixed_partial(f, point, i, j, h=1e-4):
    # d^2 f / dx_i dx_j: take the partial derivative of "the function that IS d f/d x_j"
    def df_dxj(*p):
        return partial_derivative(f, p, j, h)
    return partial_derivative(df_dxj, point, i, h)

g = lambda x, y: x**3 * y**2 + np.exp(x * y)
pt = (0.7, 1.1)
d2g_dxdy = mixed_partial(g, pt, i=0, j=1)   # d/dx ( d/dy )
d2g_dydx = mixed_partial(g, pt, i=1, j=0)   # d/dy ( d/dx )
print(f"\nClairaut check on g(x,y) = x^3 y^2 + e^(xy) at {pt}:")
print(f"  d^2g/dx dy = {d2g_dxdy:.5f}")
print(f"  d^2g/dy dx = {d2g_dydx:.5f}")
print(f"  Equal (as Clairaut's theorem guarantees for smooth g): {np.isclose(d2g_dxdy, d2g_dydx, atol=1e-3)}")

## 🤖 Why This Matters for AI

A network's loss $L(w_1, \dots, w_N)$ depends on every parameter simultaneously — GPT-3 has
175 billion of them. Training needs $\partial L/\partial w_i$ for *every single one*: exactly a
partial derivative, by definition, bundled across all $i$ into $\nabla L$, the vector
`optimizer.step()` uses. Computing each partial derivative by brute-force finite differences
needs $2N$ forward passes per step — 350 billion for GPT-3. Backpropagation computes the
*entire* gradient in roughly the cost of one extra forward pass (chain rule, from `02.04`).
Below, a tiny multi-parameter model compares brute-force partials against PyTorch's autograd.

In [ ]:
import time

import torch

# A toy "model": mean squared error of a linear function with 50 parameters, standing in
# for a (much smaller) neural network loss surface L(w_1, ..., w_50)
rng = np.random.default_rng(0)
n_params = 50
X_data = rng.standard_normal((200, n_params))
true_w = rng.standard_normal(n_params)
y_data = X_data @ true_w + rng.normal(scale=0.1, size=200)

def loss_numpy(w):
    pred = X_data @ w
    return np.mean((pred - y_data) ** 2)

w0 = rng.standard_normal(n_params)

# --- Brute-force: one partial derivative at a time, exactly like partial_derivative() above ---
t0 = time.perf_counter()
brute_grad = gradient(lambda *w: loss_numpy(np.array(w)), tuple(w0), h=1e-5)
brute_time = time.perf_counter() - t0

# --- Backprop via autograd: the WHOLE gradient in one backward pass ---
w_t = torch.tensor(w0, requires_grad=True)
X_t, y_t = torch.tensor(X_data), torch.tensor(y_data)
t0 = time.perf_counter()
loss_t = torch.mean((X_t @ w_t - y_t) ** 2)
loss_t.backward()
autograd_time = time.perf_counter() - t0

print(f"Parameters: {n_params}")
print(f"Brute-force partial derivatives: {2*n_params} function evaluations, {brute_time*1000:.2f} ms")
print(f"Autograd (backprop):             1 backward pass,        {autograd_time*1000:.2f} ms")
print(f"Gradients match: {np.allclose(brute_grad, w_t.grad.numpy(), atol=1e-3)}")
print(f"\nAt GPT-3 scale (175B params), the brute-force approach would need 350 BILLION")
print(f"forward passes for a single gradient -- backprop needs roughly one extra pass, total.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For $f(x,y,z) = xyz + x^2 z$, compute the full gradient $\nabla f$ (a 3-vector) at
   $(1, 2, 3)$ both analytically and with `gradient`. As a sanity check, confirm the shape of
   the returned gradient is a length-3 vector, matching the number of input variables, not the
   scalar output.

<details>
<summary>💡 Solution</summary>

$\frac{\partial f}{\partial x} = yz + 2xz$, $\frac{\partial f}{\partial y} = xz$,
$\frac{\partial f}{\partial z} = xy + x^2$. At $(1,2,3)$: $\partial f/\partial x = 2(3)+2(1)(3) =
6+6=12$, $\partial f/\partial y = (1)(3) = 3$, $\partial f/\partial z = (1)(2)+1^2 = 3$, so
$\nabla f(1,2,3) = (12, 3, 3)$. `gradient(f, (1,2,3))` reproduces this numerically
(`[12. 3. 3.]`, matching to well within finite-difference tolerance). The returned array has
shape `(3,)` — one entry per input coordinate — regardless of $f$ itself being scalar-valued;
the gradient's dimensionality always matches the *input* space, never the output.

</details>

### 💭 UNDERSTAND
2. At $(x,y)=(1,1)$ for $f(x,y)=x^2+y^2$, both partial derivatives equal $2$. A learner
   reasons: "since $\partial f/\partial x = \partial f/\partial y = 2$, moving from $(1,1)$ to
   $(1.01, 1.01)$ (nudging both coordinates by $0.01$) should change $f$ by about
   $2\times0.01=0.02$." Is this reasoning correct? Compute the actual change and explain.

<details>
<summary>💡 Solution</summary>

The reasoning is wrong, and the actual change is close to $2\times(2\times0.01) = 0.04$, not
$0.02$ — moving $0.01$ in *each* of two directions contributes $\partial f/\partial x \cdot
0.01$ from the $x$-move *and* $\partial f/\partial y \cdot 0.01$ from the $y$-move, and to
first order these two contributions add: $\Delta f \approx \frac{\partial f}{\partial x}\Delta x
+ \frac{\partial f}{\partial y}\Delta y = 2(0.01)+2(0.01)=0.04$. Checking numerically,
$f(1.01,1.01)-f(1,1) = 0.0402$, matching the two-term prediction (not the single-partial
$0.02$) to within the expected first-order approximation error. A single partial derivative
only tells you the rate of change from moving *one* coordinate alone; moving several
coordinates at once requires summing each one's contribution — exactly what the gradient
vector (dotted with the actual displacement) is built to do, and the reason `02.05`'s
directional derivative is $\nabla f\cdot\hat{u}$, a full dot product, rather than any single
entry of $\nabla f$ alone.

</details>

### ✏️ DERIVE
3. For a quadratic form $Q(\mathbf{w}) = \mathbf{w}^\top A\mathbf{w}$ with $A$ a **symmetric**
   $2\times2$ matrix $A=\begin{pmatrix}a&b\\b&c\end{pmatrix}$ and $\mathbf{w}=(w_1,w_2)$,
   derive $\partial Q/\partial w_1$ and $\partial Q/\partial w_2$ by first expanding $Q$ into a
   polynomial in $w_1, w_2$, then show the result equals $2(A\mathbf{w})_1$ and
   $2(A\mathbf{w})_2$ respectively — i.e. $\nabla Q = 2A\mathbf{w}$.

<details>
<summary>💡 Solution outline</summary>

Expanding: $Q(\mathbf{w}) = aw_1^2 + 2bw_1w_2 + cw_2^2$ (the two off-diagonal contributions
$bw_1w_2$ from each triangular half of $A$ combine into $2bw_1w_2$, which is exactly where
symmetry of $A$ is used). Differentiating termwise with the power and product rules:
$$\frac{\partial Q}{\partial w_1} = 2aw_1+2bw_2, \qquad \frac{\partial Q}{\partial w_2} = 2bw_1+2cw_2.$$
Separately, $A\mathbf{w} = \begin{pmatrix}aw_1+bw_2\\bw_1+cw_2\end{pmatrix}$, so
$2A\mathbf{w} = \begin{pmatrix}2aw_1+2bw_2\\2bw_1+2cw_2\end{pmatrix}$ — identical to the two
partials just computed, confirming $\nabla Q=2A\mathbf{w}$. (Numerically, at $A=\begin{pmatrix}
3&1\\1&2\end{pmatrix}$, $\mathbf{w}=(2,-1)$: $2A\mathbf{w}=(10,0)$, matching `gradient`'s
numeric output on the same $Q$ exactly.) This identity is worth memorizing outright — it's the
gradient behind every ridge-regularization penalty ($\lambda\|\mathbf{w}\|^2 = \lambda
\mathbf{w}^\top I\mathbf{w}$, gradient $2\lambda\mathbf{w}$) and every quadratic-loss curvature
term you'll meet from `02.06` onward.

</details>

### 🐍 IMPLEMENT
4. Extend `mixed_partial` into a full `hessian(f, point)` that returns the $n\times n$ matrix
   of all second partial derivatives (entry $(i,j)$ is `mixed_partial(f, point, i, j)`). Test
   it on a new function $h(x,y) = \sin(x)y^2$ at $(0.6, 1.3)$, and check both against SymPy's
   `sp.hessian` **and** that the matrix comes out symmetric, as Clairaut's theorem guarantees.

<details>
<summary>✅ How to know you're right</summary>

Your numeric Hessian should match `sp.hessian(sin(x)*y**2, (x,y))` evaluated at $(0.6,1.3)$ to
within about $10^{-3}$ (the tolerance `mixed_partial`'s own finite-difference step size,
$h=10^{-4}$, implies) — concretely, close to
$\begin{pmatrix}-1.69y^2\sin(x)... \end{pmatrix}$, or numerically approximately
$\begin{pmatrix}-0.954 & 2.146\\2.146 & 1.129\end{pmatrix}$. Separately from matching SymPy,
explicitly assert `np.allclose(H, H.T)` — symmetry isn't a coincidence of this particular
function, it's Clairaut's theorem from the Theory section applying to *any* smooth $f$, so a
correct `hessian` implementation should produce a symmetric matrix for every smooth test
function you throw at it, and a non-symmetric result (outside finite-difference tolerance)
means a bug in how `i` and `j` are being used, not a property of $h$ itself.

</details>

### 🤖 APPLY
5. Add an $\ell_2$ (ridge) penalty to a small linear-regression loss:
   $L(\mathbf{w}) = \text{MSE}(\mathbf{w}) + \lambda\|\mathbf{w}\|^2$ for 2 weights on synthetic
   data. Using `gradient`, confirm $\nabla L = \nabla(\text{MSE}) + 2\lambda\mathbf{w}$ (the
   DERIVE exercise's identity, with $A=I$), then run gradient descent to convergence at several
   values of $\lambda$ (e.g. $0, 0.5, 5$) and compare the final $\|\mathbf{w}\|$ across them.

<details>
<summary>✅ What you should observe</summary>

The numeric gradient of $L$ should match `grad(MSE) + 2*lam*w` computed separately, to within
finite-difference tolerance (around $10^{-4}$), confirming the penalty term contributes exactly
the $2\lambda\mathbf{w}$ derived in DERIVE, added on top of whatever the data term contributes.
After running gradient descent to convergence, larger $\lambda$ should produce a
*smaller*-norm final $\mathbf{w}$: in one run, $\lambda=0$ converges close to the true
generating weights ($\|\mathbf{w}\|\approx2.48$, true value $2.5$), $\lambda=0.5$ shrinks to
$\|\mathbf{w}\|\approx1.53$, and $\lambda=5$ shrinks further to $\approx0.34$ — increasing
$\lambda$ directly increases the pull of the $2\lambda\mathbf{w}$ term relative to the data
gradient at every step, which is exactly the mechanism (not just the name) behind why ridge
regression is described as "shrinking" weights toward zero. (Pushing $\lambda$ far larger, e.g.
$50$ with a learning rate tuned for smaller $\lambda$, can overshoot and diverge — a reminder
that the regularization term also changes the loss surface's curvature, a topic `02.06`
returns to.)

</details>

### 🚀 CHALLENGE
6. A single data point $(x,y)=(2,5)$ and loss $L(w_1,w_2)=(w_1x+w_2-y)^2$ (this notebook's own
   setup) has *infinitely many* exact minimizers — every $(w_1,w_2)$ on the line
   $2w_1+w_2=5$ gives $L=0$, since one equation can't pin down two unknowns. (a) Verify this
   directly by evaluating $L$ at several points on that line. (b) Run gradient descent from at
   least 4 different starting points and record which point on the line each one converges to.
   (c) Investigate what determines *which* point on the line a given start converges to, and
   connect this to the notebook's remark that $\nabla L$ "tells you exactly how each individual
   parameter should move" — here there are 2 parameters but only 1 independent equation
   constraining them.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer confirms (a) directly — $(0,5)$, $(1,3)$, $(2,1)$, $(2.5,0)$, $(-3,11)$ all give
$L=0$ exactly, since each satisfies $2w_1+w_2=5$. For (b)/(c): different starting points
converge to *different* points on the line (e.g. starting at $(0,0)$ converges to $(2,1)$,
starting at $(5,5)$ converges to $(1,3)$, starting at $(-3,2)$ converges to $(0.6,3.8)$) — and
the specific point each run lands on is exactly the **orthogonal projection of the starting
point onto the line** $2w_1+w_2=5$, matching to within numerical tolerance in every case tried.
The mechanism is that $\nabla L$ at any point is always parallel to the fixed vector $(x,1)=
(2,1)$ (the line's normal direction) — the loss only "sees" and corrects the component of
$\mathbf{w}$ along that one direction, and is completely blind to (applies zero gradient along)
the perpendicular direction, so gradient descent never moves the starting point's component
along the line at all. A strong answer connects this to the general principle: whenever a model
has more parameters than independent constraints from the data (here, 2 parameters, 1 data
point), the gradient is mathematically guaranteed to be degenerate — it has a null direction
(here, the line itself) — and *which* zero-loss solution training finds is determined by
initialization interacting with that null space, not by the loss function alone. This is a
small, fully-transparent instance of the much larger phenomenon of implicit bias/underdetermined
optima in overparameterized models trained by gradient descent.

</details>

---

## 📚 Further Reading
- Stewart, *Calculus: Early Transcendentals*, Ch. 14 (Partial Derivatives)
- Deisenroth, Faisal & Ong, *Mathematics for Machine Learning*, Ch. 5.2 (Partial Differentiation)

---

*Next notebook: [Chain Rule and Computation Graphs](04_chain_rule_and_computation_graphs.ipynb)*